# Real-time validation of ExampleObject

`ExampleObject` has one field for each kind of type that the C# to Python conversion supports. This notebook assigns good and bad values to those fields and shows what the Python class does: it **accepts** a correct value, and **refuses a wrong one at once**, when the object is created or when a field is assigned, with a message that says what is wrong.

> **This notebook tests the local version of `bhom-schema`, the code in your checkout, and not the one published on TestPyPI.** Cell 1 checks that, and stops if the package comes from `site-packages`. To use your checkout, install it once, in the environment of the notebook, with `pip install -e ".[dev]" ipykernel` from the repository folder (see section 11 of `docs/Testing-Guide.md`), choose that environment as the kernel, and restart the kernel after any change to the code.
>
> To demo a published version instead (for example on Databricks), skip the check in cell 1 and install it with `pip install --upgrade --pre bhom-schema`.

Run the cells from the top, in order. Each cell says what to expect.

## 1. Is the local version in use?

The cell prints the version and the folder the package is loaded from. It raises an error if that folder is an installed copy (`site-packages`) and not your checkout.

In [1]:
import bhom_schema
from pathlib import Path

location = Path(bhom_schema.__file__).resolve()
print("version :", bhom_schema.__version__)
print("loaded  :", location)

assert "site-packages" not in location.parts, (
    "This is an installed copy of bhom-schema, not your checkout. "
    "Run  pip install -e \".[dev]\"  in the repository folder, select that environment as the kernel, and restart the kernel."
)
print("the local version of bhom-schema is in use")

version : 10.0.0.dev0
loaded  : C:\Github\BHoM_PythonSchema\src\bhom_schema\__init__.py
the local version of bhom-schema is in use


## 2. Imports and a helper

`attempt` runs something and prints what happened: `accepted`, or `refused` with the message of the first error. A real script would let the `ValidationError` propagate.

In [2]:
from datetime import datetime, timedelta
from decimal import Decimal
from uuid import UUID

from pydantic import ValidationError

from bhom_schema.DataSchema_oM.ExampleObject import ExampleObject
from bhom_schema.DataSchema_oM.ExampleStatus import ExampleStatus
from bhom_schema.DataSchema_oM.ExampleFinish import ExampleFinish
from bhom_schema.DataSchema_oM.Enums.StructuralMaterial import StructuralMaterial
from bhom_schema.DataSchema_oM.Grid import Grid
from bhom_schema.DataSchema_oM.Storey import Storey

refused = []

def attempt(label, action):
    """Runs the action; prints accepted, or refused with the first error, and records the refusals."""
    try:
        action()
        print(f"accepted  {label}")
    except ValidationError as error:
        first = error.errors()[0]
        field = ".".join(str(part) for part in first["loc"])
        print(f"refused   {label}\n          {field}: {first['msg']}")
        refused.append(label)

## 3. A valid object

Every field has a default, so `ExampleObject()` is valid. Here a few fields are set. **Expect:** `accepted`, then the values.

In [3]:
example = ExampleObject(
    SomeStringProperty="beam B1",
    SomeNumberProperty=3,
    Cost=Decimal("19.99"),
    Status=ExampleStatus.Approved,
    Finish=ExampleFinish.Painted | ExampleFinish.Insulated,
    Labels=["steel", "level 2"],
    Lookup={"span": 6.5},
)
attempt("a valid object", lambda: ExampleObject(Status=ExampleStatus.Approved))
print(example.Status, example.Finish, example.Cost)

accepted  a valid object
ExampleStatus.Approved 3 19.99


## 4. Wrong types are refused

A text where a number is expected, a number with a decimal part for a whole number, and a text that is not a date. **Expect:** three `refused` lines naming the field.

In [4]:
attempt("SomeNumberProperty = 'three'", lambda: ExampleObject(SomeNumberProperty="three"))
attempt("SomeNumberProperty = 2.5 (not a whole number)", lambda: ExampleObject(SomeNumberProperty=2.5))
attempt("CreatedOn = 'yesterday'", lambda: ExampleObject(CreatedOn="yesterday"))
attempt("ExternalId = 'not-a-guid'", lambda: ExampleObject(ExternalId="not-a-guid"))
attempt("IsActive = 'maybe'", lambda: ExampleObject(IsActive="maybe"))

refused   SomeNumberProperty = 'three'
          SomeNumberProperty: Input should be a valid integer, unable to parse string as an integer
refused   SomeNumberProperty = 2.5 (not a whole number)
          SomeNumberProperty: Input should be a valid integer, got a number with a fractional part
refused   CreatedOn = 'yesterday'
          CreatedOn: Input should be a valid datetime or date, input is too short
refused   ExternalId = 'not-a-guid'
          ExternalId: Input should be a valid UUID, invalid character: found `n` at 0
refused   IsActive = 'maybe'
          IsActive: Input should be a valid boolean, unable to interpret input


## 5. Enum members are checked

`Status` can only be `None`, `Draft`, `Approved` or `Retired`. A value that is not a member is refused with the list of members, and a near miss gets a suggestion. **Expect:** the first line `accepted`; the others `refused` with `Members: ...`, and `Did you mean Approved?` for the typo and the wrong case.

In [5]:
attempt("Status = 'Approved' (a member name)", lambda: ExampleObject(Status="Approved"))
attempt("Status = 'Published' (not a member)", lambda: ExampleObject(Status="Published"))
attempt("Status = 'Aproved' (a typo)", lambda: ExampleObject(Status="Aproved"))
attempt("Status = 'approved' (wrong case)", lambda: ExampleObject(Status="approved"))
attempt("Material = 'Unobtainium' (an optional enum)", lambda: ExampleObject(Material="Unobtainium"))
attempt("Material = None (optional, so allowed)", lambda: ExampleObject(Material=None))

accepted  Status = 'Approved' (a member name)
refused   Status = 'Published' (not a member)
          Status: Published is not a member of ExampleStatus. Members: None, Draft, Approved, Retired.
refused   Status = 'Aproved' (a typo)
          Status: Aproved is not a member of ExampleStatus. Members: None, Draft, Approved, Retired. Did you mean Approved?
refused   Status = 'approved' (wrong case)
          Status: approved is not a member of ExampleStatus. Members: None, Draft, Approved, Retired. Did you mean Approved?
refused   Material = 'Unobtainium' (an optional enum)
          Material: Unobtainium is not a member of StructuralMaterial. Members: SteelHotRolled, SteelLightGauge, Composite, ConcreteInSitu, ConcretePostTension, ConcretePrecast, TimberGlulam, TimberLVL, TimberCLT, TimberSawn, TimberGeneral, MasonryConcreteBlockwork, MasonryBrickwork, Other.
accepted  Material = None (optional, so allowed)


## 6. Flags: combinations are checked

`Finish` can hold several members, written as names separated by a comma. **Expect:** the first two `accepted`; the unknown name and the number with a bit that no member defines `refused`.

In [6]:
attempt("Finish = 'Painted, Insulated'", lambda: ExampleObject(Finish="Painted, Insulated"))
attempt("Finish = Painted | Waterproofed", lambda: ExampleObject(Finish=ExampleFinish.Painted | ExampleFinish.Waterproofed))
attempt("Finish = 'Painted, Glittery'", lambda: ExampleObject(Finish="Painted, Glittery"))
attempt("Finish = 8 (no member has this bit)", lambda: ExampleObject(Finish=8))

print(ExampleObject(Finish="Painted, Insulated").model_dump()["Finish"])

accepted  Finish = 'Painted, Insulated'
accepted  Finish = Painted | Waterproofed
refused   Finish = 'Painted, Glittery'
          Finish: Glittery is not a member of ExampleFinish. Members: None, Painted, Insulated, Waterproofed. Combine names with ', '.
refused   Finish = 8 (no member has this bit)
          Finish: 8 has bits that no member defines in ExampleFinish. Members: None, Painted, Insulated, Waterproofed. Combine names with ', '.
Painted, Insulated


## 7. Assigning a field is checked too

The check runs when you change a field of an existing object, not only when you create it. This is what catches a mistake in a notebook cell like `example.Status = "typo"`. **Expect:** the good assignment `accepted` and the object changed; the bad ones `refused`, and the object unchanged.

In [7]:
def assign(name, value):
    """Assigns a field of the example object."""
    setattr(example, name, value)

attempt("example.Status = 'Retired'", lambda: assign("Status", "Retired"))
print("  Status is now", example.Status)

attempt("example.Status = 'Retird'", lambda: assign("Status", "Retird"))
attempt("example.SomeNumberProperty = 'x'", lambda: assign("SomeNumberProperty", "x"))
print("  Status is still", example.Status, "and SomeNumberProperty is still", example.SomeNumberProperty)

accepted  example.Status = 'Retired'
  Status is now ExampleStatus.Retired
refused   example.Status = 'Retird'
          Status: Retird is not a member of ExampleStatus. Members: None, Draft, Approved, Retired. Did you mean Retired?
refused   example.SomeNumberProperty = 'x'
          SomeNumberProperty: Input should be a valid integer, unable to parse string as an integer
  Status is still ExampleStatus.Retired and SomeNumberProperty is still 3


## 8. Collections: the items are checked

A list of texts, an array of numbers, a matrix of whole numbers, a set, dictionaries with a text key and with an enum key, and a pair. **Expect:** all `refused` except the first, which is a correct dictionary with an enum key; the message says which item is wrong.

In [8]:
attempt("TotalsByStatus = {Draft: 1.5}", lambda: ExampleObject(TotalsByStatus={ExampleStatus.Draft: 1.5}))
attempt("TotalsByStatus = {'Bogus': 1.5} (key not a member)", lambda: ExampleObject(TotalsByStatus={"Bogus": 1.5}))
attempt("Labels = ['a', 5] (an item is not a text)", lambda: ExampleObject(Labels=["a", 5]))
attempt("Values = [1.0, 'x']", lambda: ExampleObject(Values=[1.0, "x"]))
attempt("Matrix = [[1, 2], [3, 'x']]", lambda: ExampleObject(Matrix=[[1, 2], [3, "x"]]))
attempt("Lookup = {'span': 'long'} (value not a number)", lambda: ExampleObject(Lookup={"span": "long"}))
attempt("Pair = (1, 2) (second item not a text)", lambda: ExampleObject(Pair=(1, 2)))
attempt("Pair = (1, 'a', 'b') (too many items)", lambda: ExampleObject(Pair=(1, "a", "b")))

accepted  TotalsByStatus = {Draft: 1.5}
refused   TotalsByStatus = {'Bogus': 1.5} (key not a member)
          TotalsByStatus.Bogus.[key]: Bogus is not a member of ExampleStatus. Members: None, Draft, Approved, Retired.
refused   Labels = ['a', 5] (an item is not a text)
          Labels.1: Input should be a valid string
refused   Values = [1.0, 'x']
          Values.1: Input should be a valid number, unable to parse string as a number
refused   Matrix = [[1, 2], [3, 'x']]
          Matrix.1.1: Input should be a valid integer, unable to parse string as an integer
refused   Lookup = {'span': 'long'} (value not a number)
          Lookup.span: Input should be a valid number, unable to parse string as a number
refused   Pair = (1, 2) (second item not a text)
          Pair.1: Input should be a valid string
refused   Pair = (1, 'a', 'b') (too many items)
          Pair: Tuple should have at most 2 items after validation, not 3


## 9. Other objects: they are checked inside

`MainGrid` must be a `Grid` (or a dictionary that reads as one), `Storeys` a list of `Storey`, and `Anything` any BHoM object. **Expect:** `accepted` for the good ones, `refused` for the others, with the path to the wrong field (for example `Storeys.1.TypicalHeight`).

In [9]:
attempt("MainGrid = Grid(...)", lambda: ExampleObject(MainGrid=Grid(TypicalXSpacing=8.0)))
attempt("MainGrid = {'TypicalXSpacing': 'wide'}", lambda: ExampleObject(MainGrid={"TypicalXSpacing": "wide"}))
attempt("Storeys = [Storey, Storey with a bad height]", lambda: ExampleObject(Storeys=[Storey(TypicalHeight=3.5), {"TypicalHeight": "tall"}]))
attempt("Anything = a Storey", lambda: ExampleObject(Anything=Storey(TypicalHeight=3.0)))
attempt("Anything = 5 (not an object)", lambda: ExampleObject(Anything=5))
attempt("Extra = anything at all", lambda: ExampleObject(Extra={"any": ["thing", 1]}))

accepted  MainGrid = Grid(...)
refused   MainGrid = {'TypicalXSpacing': 'wide'}
          MainGrid.TypicalXSpacing: Input should be a valid number, unable to parse string as a number
refused   Storeys = [Storey, Storey with a bad height]
          Storeys.1.TypicalHeight: Input should be a valid number, unable to parse string as a number
accepted  Anything = a Storey
refused   Anything = 5 (not an object)
          Anything: expected an object of IObject or a JSON object with _t
accepted  Extra = anything at all


## 10. JSON read from a file or a table is checked as it is read

Data from another system (BHoM C# writes the same form) is validated when it is read. The first text is correct; the second has a wrong enum member and a wrong type in one object, and **both errors are reported together**. **Expect:** `accepted`, then `refused`, then `2 errors`.

In [10]:
good = '{"_t": "BH.oM.Adapters.DataSchema.ExampleObject", "Status": "Retired", "Finish": "Painted, Waterproofed", "TotalsByStatus": [{"k": "Draft", "v": 2.5}]}'
bad = '{"_t": "BH.oM.Adapters.DataSchema.ExampleObject", "Status": "Published", "SomeNumberProperty": "many"}'

attempt("good JSON", lambda: ExampleObject.model_validate_json(good))
attempt("bad JSON", lambda: ExampleObject.model_validate_json(bad))

try:
    ExampleObject.model_validate_json(bad)
except ValidationError as error:
    print(f"{error.error_count()} errors:")
    for item in error.errors():
        print("  ", item["loc"][0], "-", item["msg"][:90])

accepted  good JSON
refused   bad JSON
          SomeNumberProperty: Input should be a valid integer, unable to parse string as an integer
2 errors:
   SomeNumberProperty - Input should be a valid integer, unable to parse string as an integer
   Status - Published is not a member of ExampleStatus. Members: None, Draft, Approved, Retired.


## 11. Write it back, and read it again

A valid object is written as JSON and read back to an equal object. **Expect:** `True`.

In [11]:
full = ExampleObject(
    SomeStringProperty="demo",
    Status=ExampleStatus.Approved,
    Material=StructuralMaterial.Other,
    Finish="Painted, Insulated",
    Labels=["a", "b"],
    TotalsByStatus={ExampleStatus.Draft: 1.0, ExampleStatus.None_: 2.0},
    Pair=(1, "one"),
    MainGrid=Grid(TypicalXSpacing=8.0),
    Storeys=[Storey(TypicalHeight=3.5)],
    UniqueNames={"x", "y"},
)
text = full.model_dump_json()
print(text[:260], "...")
print(ExampleObject.model_validate_json(text) == full)

{"_t":"BH.oM.Adapters.DataSchema.ExampleObject","BHoM_Guid":"76546acd-84c9-4468-a781-67738f856448","Name":"","Fragments":[],"Tags":[],"CustomData":{},"SomeStringProperty":"demo","SomeNumberProperty":0,"SomeDecimalNumber":1.5,"IsActive":true,"Cost":"10.25","Let ...
True


## 12. Summary

Counts the refusals above. **Expect:** `All checks passed`. If a number differs, the version of the package does not validate as it should: report the cell.

In [12]:
expected_refusals = 24
assert len(refused) == expected_refusals, f"{len(refused)} values were refused, expected {expected_refusals}"
assert ExampleObject.model_validate_json(text) == full
print(f"{len(refused)} wrong values were refused, and the correct ones were accepted")
print("All checks passed")

24 wrong values were refused, and the correct ones were accepted
All checks passed
